# 06 — Validasi Financial Impact TP / TN / FP / FN

## Tujuan notebook

Notebook ini dibuat sebagai **audit trail / bukti reproduksibilitas** untuk angka financial impact pada dokumen **“Breakdown Financial Impact TP / TN / FP / FN — Merchant Acquisition Product Recommendation”**.

Prinsip notebook ini:

1. **Tidak meng-hardcode angka turunan** seperti Rp397.819, Rp27.937, Rp875.324, atau Rp44.472.710.
2. Economics merchant **direkonstruksi ulang dari transaksi C2M** dan variabel merchant yang dihasilkan pipeline.
3. Product recommendation dan model metadata dibaca dari artefak Notebook 04.
4. Precision holdout direproduksi kembali dari data berlabel dengan konfigurasi model yang tersimpan.
5. Formula cost, contribution, TP/TN/FP/FN diterapkan kembali secara transparan.
6. Pada akhir notebook, hasil kalkulasi dibandingkan dengan angka referensi pada PDF.

> **Catatan:** angka PDF merepresentasikan versi artefak yang menghasilkan 246 target non-BNI, 105 rekomendasi QRIS, dan 141 rekomendasi QRIS + EDC. Jika model di-refit setelah perubahan metodologi sehingga rekomendasi berubah, notebook ini tetap menghitung angka terbaru dan akan menunjukkan perbedaannya terhadap referensi PDF, bukan memaksa angka lama.

## 1. Peta sumber data dan formula

Alur pembuktian di notebook ini:

```text
transactions_clean.csv + merchants_clean.csv
        ↓
rekonstruksi Merchant MDR / FBI Profile
        ↓
monthly_net_qris_fbi + monthly_net_edc_fbi

feature_table.csv + product_rec_metadata.pkl
        ↓
reproduksi holdout product model
        ↓
precision QRIS + precision QRIS+EDC

merchants_with_product_rec.csv
        ↓
jumlah target + kelompok rekomendasi

MDR/FBI + rekomendasi + biaya produk
        ↓
contribution per merchant
        ↓
portfolio TP / TN / FP / FN approximation
        ↓
Expected Decision Value
```

**Separation of concern:** Notebook 01 menyediakan building block MDR/FBI, Notebook 04 menyediakan product recommendation dan performance model, sedangkan formula contribution/risk baru mempunyai arti setelah recommendation diketahui.

In [1]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
TEST_SIZE = 0.20


def find_project_root():
    """Cari root repo baik notebook dijalankan dari root maupun folder notebooks."""
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "processed").exists() and (candidate / "models").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Project root tidak ditemukan. Jalankan notebook dari dalam repository "
        "yang memiliki folder data/processed dan models."
    )


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\DELL\OneDrive\Desktop\odp_python\capstone_merchant


## 2. Load artefak yang benar-benar dihasilkan pipeline

Tidak ada angka portfolio yang dimasukkan manual pada tahap ini. Notebook membaca:

- `transactions_clean.csv` — transaksi yang sudah dibersihkan Notebook 01;
- `merchants_clean.csv` — merchant master hasil cleaning;
- `merchant_mdr_profile.csv` — output economics Notebook 01, digunakan sebagai pembanding;
- `feature_table.csv` — feature graph/profile untuk model;
- `merchants_with_product_rec.csv` — hasil rekomendasi produk Notebook 04;
- `product_rec_metadata.pkl` — konfigurasi + holdout metrics model Notebook 04.

In [2]:
trx_clean = pd.read_csv(DATA_DIR / "transactions_clean.csv")
merchants_clean = pd.read_csv(DATA_DIR / "merchants_clean.csv")
mdr_saved = pd.read_csv(DATA_DIR / "merchant_mdr_profile.csv")
feature_table = pd.read_csv(DATA_DIR / "feature_table.csv")
product_rec = pd.read_csv(DATA_DIR / "merchants_with_product_rec.csv")
product_metadata = joblib.load(MODEL_DIR / "product_rec_metadata.pkl")

trx_clean["timestamp_dt"] = pd.to_datetime(trx_clean["timestamp_dt"])

artifact_summary = pd.DataFrame({
    "Artefak": [
        "transactions_clean.csv",
        "merchants_clean.csv",
        "merchant_mdr_profile.csv",
        "feature_table.csv",
        "merchants_with_product_rec.csv",
    ],
    "Rows": [
        len(trx_clean), len(merchants_clean), len(mdr_saved),
        len(feature_table), len(product_rec)
    ],
})

display(artifact_summary)
print("Model metadata version :", product_metadata.get("version"))
print("Model name             :", product_metadata.get("model_name"))
print("Balancing strategy     :", product_metadata.get("balancing_strategy"))
print("Best params            :", product_metadata.get("best_params"))

,Artefak,Rows
0,transactions_clean.csv,65676
1,merchants_clean.csv,708
2,merchant_mdr_profile.csv,708
3,feature_table.csv,708
4,merchants_with_product_rec.csv,708


Model metadata version : pipeline_v2
Model name             : Logistic Regression
Balancing strategy     : SMOTENC
Best params            : {'model__C': 1.0, 'model__penalty': 'l1'}


## 3. Rekonstruksi Merchant MDR Profile langsung dari transaksi

Bagian ini mengulang building block economics dari Notebook 01.

### Asumsi yang digunakan

- Net FBI retained share: **85% dari Gross MDR**.
- QRIS UMI:
  - transaksi ≤ Rp500.000 → MDR 0%;
  - transaksi > Rp500.000 → MDR 0,3%.
- QRIS non-UMI → MDR 0,7%.
- EDC debit → MDR 0,15%.
- EDC credit card → MDR 2%.
- `bank_transfer` / `unknown` tidak diberi acquiring MDR pada scope ini.

Gross MDR dihitung **per transaksi**, kemudian dijumlahkan per merchant dan dibagi observation window untuk memperoleh rata-rata bulanan.

In [3]:
NET_FBI_SHARE = 0.85
MDR_QRIS_UMI_HIGH = 0.003
MDR_QRIS_REGULAR = 0.007
MDR_EDC_DEBIT = 0.0015
MDR_EDC_CREDIT = 0.02

trx_econ = trx_clean[trx_clean["trx_type"].eq("C2M")].copy()
trx_econ = trx_econ.merge(
    merchants_clean[["merchant_id", "skala_usaha"]],
    left_on="target_id",
    right_on="merchant_id",
    how="left",
    validate="many_to_one",
)

qris_rate = np.where(
    trx_econ["payment_type"].eq("QRIS"),
    np.where(
        trx_econ["skala_usaha"].eq("UMI"),
        np.where(
            trx_econ["amount"].le(500_000),
            0.0,
            MDR_QRIS_UMI_HIGH,
        ),
        MDR_QRIS_REGULAR,
    ),
    0.0,
)

edc_rate = np.select(
    [
        trx_econ["payment_type"].eq("debit"),
        trx_econ["payment_type"].eq("credit_card"),
    ],
    [MDR_EDC_DEBIT, MDR_EDC_CREDIT],
    default=0.0,
)

trx_econ["qris_rate"] = qris_rate
trx_econ["edc_rate"] = edc_rate
trx_econ["gross_qris_mdr"] = trx_econ["amount"] * trx_econ["qris_rate"]
trx_econ["gross_edc_mdr"] = trx_econ["amount"] * trx_econ["edc_rate"]
trx_econ["qris_amount"] = np.where(
    trx_econ["payment_type"].eq("QRIS"), trx_econ["amount"], 0.0
)
trx_econ["edc_amount"] = np.where(
    trx_econ["payment_type"].isin(["debit", "credit_card"]),
    trx_econ["amount"],
    0.0,
)

period_start = trx_econ["timestamp_dt"].min().to_period("M")
period_end = trx_econ["timestamp_dt"].max().to_period("M")
OBSERVATION_MONTHS = (
    (period_end.year - period_start.year) * 12
    + (period_end.month - period_start.month)
    + 1
)

agg = (
    trx_econ.groupby("target_id")
    .agg(
        observed_qris_amount=("qris_amount", "sum"),
        observed_edc_amount=("edc_amount", "sum"),
        gross_qris_mdr_period=("gross_qris_mdr", "sum"),
        gross_edc_mdr_period=("gross_edc_mdr", "sum"),
    )
    .reset_index()
    .rename(columns={"target_id": "merchant_id"})
)

mdr_rebuilt = merchants_clean[["merchant_id"]].merge(
    agg, on="merchant_id", how="left", validate="one_to_one"
)
num_cols = [c for c in mdr_rebuilt.columns if c != "merchant_id"]
mdr_rebuilt[num_cols] = mdr_rebuilt[num_cols].fillna(0.0)

mdr_rebuilt["observation_months"] = OBSERVATION_MONTHS
mdr_rebuilt["monthly_gross_qris_mdr"] = (
    mdr_rebuilt["gross_qris_mdr_period"] / OBSERVATION_MONTHS
)
mdr_rebuilt["monthly_gross_edc_mdr"] = (
    mdr_rebuilt["gross_edc_mdr_period"] / OBSERVATION_MONTHS
)
mdr_rebuilt["monthly_net_qris_fbi"] = (
    mdr_rebuilt["monthly_gross_qris_mdr"] * NET_FBI_SHARE
)
mdr_rebuilt["monthly_net_edc_fbi"] = (
    mdr_rebuilt["monthly_gross_edc_mdr"] * NET_FBI_SHARE
)
mdr_rebuilt["monthly_net_total_fbi"] = (
    mdr_rebuilt["monthly_net_qris_fbi"]
    + mdr_rebuilt["monthly_net_edc_fbi"]
)

print(
    f"Observation window: {period_start} s.d. {period_end} "
    f"({OBSERVATION_MONTHS} bulan)"
)
print(f"C2M transaction rows used: {len(trx_econ):,}")

display(
    mdr_rebuilt[
        [
            "monthly_gross_qris_mdr",
            "monthly_gross_edc_mdr",
            "monthly_net_qris_fbi",
            "monthly_net_edc_fbi",
            "monthly_net_total_fbi",
        ]
    ].describe().T
)

Observation window: 2025-07 s.d. 2026-06 (12 bulan)
C2M transaction rows used: 49,066


,count,mean,std,min,25%,50%,75%,max
monthly_gross_qris_mdr,708.0000,"142,821.9138","126,789.4934",0.0000,"1,524.9656","163,699.1268","238,508.0113","625,027.2921"
monthly_gross_edc_mdr,708.0000,"288,463.5957","399,242.7291",54.2234,"4,527.7627","178,939.3254","387,041.4147","3,059,161.3020"
monthly_net_qris_fbi,708.0000,"121,398.6267","107,771.0694",0.0000,"1,296.2208","139,144.2578","202,731.8096","531,273.1983"
monthly_net_edc_fbi,708.0000,"245,194.0564","339,356.3197",46.0899,"3,848.5983","152,098.4266","328,985.2025","2,600,287.1067"
monthly_net_total_fbi,708.0000,"366,592.6831","420,311.4282",688.4777,"5,180.3282","307,291.9237","540,035.6881","3,088,729.2488"


### 3.1 Validasi terhadap output `merchant_mdr_profile.csv`

Tujuannya memastikan angka economics yang dipakai downstream memang dapat dibentuk ulang dari transaksi, bukan angka yang dimasukkan manual.

In [4]:
compare_cols = [
    "observed_qris_amount",
    "observed_edc_amount",
    "gross_qris_mdr_period",
    "gross_edc_mdr_period",
    "monthly_gross_qris_mdr",
    "monthly_gross_edc_mdr",
    "monthly_net_qris_fbi",
    "monthly_net_edc_fbi",
    "monthly_net_total_fbi",
]

mdr_check = mdr_saved[["merchant_id"] + compare_cols].merge(
    mdr_rebuilt[["merchant_id"] + compare_cols],
    on="merchant_id",
    how="outer",
    suffixes=("_saved", "_rebuilt"),
    validate="one_to_one",
)

validation_rows = []
for col in compare_cols:
    diff = (
        mdr_check[f"{col}_saved"] - mdr_check[f"{col}_rebuilt"]
    ).abs()
    validation_rows.append({
        "Variable": col,
        "Max absolute difference": diff.max(),
        "Status": "PASS" if np.allclose(
            mdr_check[f"{col}_saved"],
            mdr_check[f"{col}_rebuilt"],
            rtol=1e-9,
            atol=1e-6,
            equal_nan=True,
        ) else "CHECK",
    })

mdr_validation = pd.DataFrame(validation_rows)
display(mdr_validation)

assert (mdr_validation["Status"] == "PASS").all(), (
    "Rekonstruksi MDR/FBI berbeda dari merchant_mdr_profile.csv. "
    "Pastikan Notebook 01 sudah dirun dengan data transaksi terbaru."
)
print("PASS — Merchant MDR/FBI Profile berhasil direkonstruksi dari transaksi.")

,Variable,Max absolute difference,Status
0,observed_qris_amount,0.0000,PASS
1,observed_edc_amount,0.0000,PASS
2,gross_qris_mdr_period,0.0000,PASS
3,gross_edc_mdr_period,0.0000,PASS
4,monthly_gross_qris_mdr,0.0000,PASS
5,monthly_gross_edc_mdr,0.0000,PASS
6,monthly_net_qris_fbi,0.0000,PASS
7,monthly_net_edc_fbi,0.0000,PASS
8,monthly_net_total_fbi,0.0000,PASS


PASS — Merchant MDR/FBI Profile berhasil direkonstruksi dari transaksi.


## 4. Drill-down: bukti transaksi → MDR pada satu merchant

Cell ini mengambil satu merchant non-BNI dengan Net FBI terbesar sebagai contoh. Rate dan Gross MDR ditampilkan per transaksi sehingga sumber angka dapat ditelusuri sampai baris transaksi.

In [5]:
non_bni_ids = set(
    product_rec.loc[product_rec["is_bni_acquiring"].eq("Tidak"), "merchant_id"]
)

sample_id = (
    mdr_rebuilt[mdr_rebuilt["merchant_id"].isin(non_bni_ids)]
    .sort_values("monthly_net_total_fbi", ascending=False)
    .iloc[0]["merchant_id"]
)

sample_merchant = merchants_clean.loc[
    merchants_clean["merchant_id"].eq(sample_id),
    ["merchant_id", "nama", "skala_usaha", "kategori", "kota"],
]

display(sample_merchant)

sample_trx = trx_econ.loc[
    trx_econ["target_id"].eq(sample_id),
    [
        "trx_id", "timestamp_dt", "payment_type", "amount",
        "qris_rate", "edc_rate", "gross_qris_mdr", "gross_edc_mdr"
    ],
].copy()

sample_trx["total_gross_mdr"] = (
    sample_trx["gross_qris_mdr"] + sample_trx["gross_edc_mdr"]
)

display(sample_trx.sort_values("timestamp_dt").head(15))

sample_summary = mdr_rebuilt.loc[
    mdr_rebuilt["merchant_id"].eq(sample_id),
    [
        "merchant_id", "observed_qris_amount", "observed_edc_amount",
        "gross_qris_mdr_period", "gross_edc_mdr_period",
        "monthly_net_qris_fbi", "monthly_net_edc_fbi",
        "monthly_net_total_fbi"
    ],
]
display(sample_summary)

,merchant_id,nama,skala_usaha,kategori,kota
588,M0580,PT Buana Komputer Sentosa,UBE,Electronics,Bandung


,trx_id,timestamp_dt,payment_type,amount,qris_rate,edc_rate,gross_qris_mdr,gross_edc_mdr,total_gross_mdr
31688,T046050,2025-07-02 05:53:05,bank_transfer,"27,687,780.7400",0.0000,0.0000,0.0000,0.0000,0.0000
3232,T004696,2025-07-05 07:08:52,credit_card,"43,339,971.6200",0.0000,0.0200,0.0000,"866,799.4324","866,799.4324"
31751,T046133,2025-07-08 15:29:56,QRIS,"21,734,317.8400",0.0070,0.0000,"152,140.2249",0.0000,"152,140.2249"
11857,T017285,2025-07-10 20:13:59,debit,"48,805,928.5000",0.0000,0.0015,0.0000,"73,208.8927","73,208.8927"
46535,T067625,2025-07-11 11:11:09,debit,"49,533,349.6200",0.0000,0.0015,0.0000,"74,300.0244","74,300.0244"
36378,T052900,2025-07-12 08:10:23,QRIS,"33,136,355.0700",0.0070,0.0000,"231,954.4855",0.0000,"231,954.4855"
40796,T059347,2025-07-12 12:34:03,bank_transfer,"26,528,580.1800",0.0000,0.0000,0.0000,0.0000,0.0000
39847,T057983,2025-07-19 17:26:47,bank_transfer,"38,148,966.1600",0.0000,0.0000,0.0000,0.0000,0.0000
18226,T026615,2025-07-21 06:18:15,debit,"103,940,060.8600",0.0000,0.0015,0.0000,"155,910.0913","155,910.0913"
19686,T028717,2025-07-22 00:18:21,debit,"44,360,304.7000",0.0000,0.0015,0.0000,"66,540.4571","66,540.4571"


,merchant_id,observed_qris_amount,observed_edc_amount,gross_qris_mdr_period,gross_edc_mdr_period,monthly_net_qris_fbi,monthly_net_edc_fbi,monthly_net_total_fbi
588,M0580,"461,407,384.6200","2,617,005,392.6200","3,229,851.6923","26,572,847.4479","228,781.1615","1,882,243.3609","2,111,024.5224"


## 5. Reproduksi precision holdout Product Recommendation

PDF menggunakan precision per predicted class sebagai **portfolio error rate**. Agar angka 72% dan 84% tidak menjadi input manual, cell berikut mereproduksi holdout model menggunakan:

- `feature_table.csv`;
- split yang sama (`random_state=42`, stratified, 20% holdout);
- model family, balancing strategy, dan best parameter yang tersimpan di `product_rec_metadata.pkl`.

Ini **bukan tuning ulang**; hanya reproduksi model evaluasi dengan konfigurasi final yang sudah dihasilkan Notebook 04.

In [6]:
from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


def assign_product_label(product):
    if pd.isna(product):
        return np.nan
    product_upper = str(product).upper()
    if "EDC" in product_upper:
        return 1
    if "QRIS" in product_upper:
        return 0
    return np.nan


PRODUCT_FEATURES = list(product_metadata["feature_columns"])
CATEGORICAL_FEATURES = [c for c in ["kategori", "kota"] if c in PRODUCT_FEATURES]
NUMERIC_FEATURES = [c for c in PRODUCT_FEATURES if c not in CATEGORICAL_FEATURES]
NUMERIC_INDICES = list(range(len(NUMERIC_FEATURES)))
CATEGORICAL_INDICES = list(
    range(len(NUMERIC_FEATURES), len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES))
)

labeled = feature_table.loc[feature_table["is_bni_acquiring"].eq("Ya")].copy()
labeled["product_label"] = labeled["produk_bni"].apply(assign_product_label)
labeled = labeled.dropna(subset=["product_label"]).copy()
labeled["product_label"] = labeled["product_label"].astype(int)

X = labeled[PRODUCT_FEATURES].copy()
y = labeled["product_label"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)


def make_onehot_preprocessor():
    numeric_transformer = SklearnPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = SklearnPipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer(
        [
            ("numeric", numeric_transformer, NUMERIC_FEATURES),
            ("categorical", categorical_transformer, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
        sparse_threshold=0,
        verbose_feature_names_out=False,
    )


def make_smotenc_preprocessor():
    numeric_transformer = SklearnPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = SklearnPipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer(
        [
            ("numeric", numeric_transformer, NUMERIC_FEATURES),
            ("categorical", categorical_transformer, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
        sparse_threshold=0,
        verbose_feature_names_out=False,
    )


def make_smotenc_postprocessor():
    return ColumnTransformer(
        [
            ("numeric", "passthrough", NUMERIC_INDICES),
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                CATEGORICAL_INDICES,
            ),
        ],
        remainder="drop",
        sparse_threshold=0,
        verbose_feature_names_out=False,
    )


def build_base_model(name):
    if name == "Logistic Regression":
        return LogisticRegression(
            max_iter=3_000,
            solver="liblinear",
            random_state=RANDOM_STATE,
        )
    if name == "Random Forest":
        return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)
    if name == "XGBoost":
        from xgboost import XGBClassifier
        return XGBClassifier(
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
    raise ValueError(f"Model belum didukung validation notebook: {name}")


def make_evaluation_pipeline(model, sampler_name):
    if sampler_name == "SMOTENC":
        return Pipeline([
            ("preprocess", make_smotenc_preprocessor()),
            (
                "balance",
                SMOTENC(
                    categorical_features=CATEGORICAL_INDICES,
                    random_state=RANDOM_STATE,
                    k_neighbors=5,
                ),
            ),
            ("postprocess", make_smotenc_postprocessor()),
            ("model", model),
        ])

    sampler = (
        SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
        if sampler_name == "SMOTE"
        else "passthrough"
    )
    return Pipeline([
        ("preprocess", make_onehot_preprocessor()),
        ("balance", sampler),
        ("model", model),
    ])


model_name = product_metadata["model_name"]
balancing_strategy = product_metadata["balancing_strategy"]
best_params = dict(product_metadata["best_params"])

holdout_pipeline = make_evaluation_pipeline(
    build_base_model(model_name),
    balancing_strategy,
)
holdout_pipeline.set_params(**best_params)
holdout_pipeline.fit(X_train, y_train)

y_pred = holdout_pipeline.predict(X_test)
y_proba = holdout_pipeline.predict_proba(X_test)[:, 1]

precision_qris_exact = precision_score(y_test, y_pred, pos_label=0, zero_division=0)
precision_edc_exact = precision_score(y_test, y_pred, pos_label=1, zero_division=0)

holdout_metrics_rebuilt = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision_qris": precision_qris_exact,
    "precision_edc": precision_edc_exact,
    "recall_edc": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
    "f1_edc": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
    "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba),
}

display(pd.Series(holdout_metrics_rebuilt, name="Reproduced holdout").to_frame())

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["Actual QRIS", "Actual QRIS + EDC"],
    columns=["Pred QRIS", "Pred QRIS + EDC"],
)
display(cm_df)

metadata_metrics = product_metadata.get("test_metrics", {})
metric_check = pd.DataFrame([
    {
        "Metric": "precision_edc",
        "Metadata": metadata_metrics.get("precision_edc"),
        "Reproduced": precision_edc_exact,
    },
    {
        "Metric": "macro_f1",
        "Metadata": metadata_metrics.get("macro_f1"),
        "Reproduced": holdout_metrics_rebuilt["macro_f1"],
    },
    {
        "Metric": "roc_auc",
        "Metadata": metadata_metrics.get("roc_auc"),
        "Reproduced": holdout_metrics_rebuilt["roc_auc"],
    },
])
metric_check["Absolute diff"] = (
    metric_check["Metadata"] - metric_check["Reproduced"]
).abs()
metric_check["Status"] = np.where(metric_check["Absolute diff"] < 1e-10, "PASS", "CHECK")
display(metric_check)

PRECISION_QRIS_PRESENTATION = round(float(precision_qris_exact), 2)
PRECISION_EDC_PRESENTATION = round(float(precision_edc_exact), 2)

print(f"Precision QRIS exact          : {precision_qris_exact:.6f}")
print(f"Precision QRIS presentation   : {PRECISION_QRIS_PRESENTATION:.0%}")
print(f"Precision QRIS+EDC exact      : {precision_edc_exact:.6f}")
print(f"Precision QRIS+EDC presentation: {PRECISION_EDC_PRESENTATION:.0%}")

,Reproduced holdout
accuracy,0.7634
balanced_accuracy,0.7686
precision_qris,0.7059
precision_edc,0.8333
recall_edc,0.7000
f1_edc,0.7609
macro_f1,0.7634
roc_auc,0.8372


,Pred QRIS,Pred QRIS + EDC
Actual QRIS,36,7
Actual QRIS + EDC,15,35


,Metric,Metadata,Reproduced,Absolute diff,Status
0,precision_edc,0.8333,0.8333,0.0000,PASS
1,macro_f1,0.7634,0.7634,0.0000,PASS
2,roc_auc,0.8372,0.8372,0.0000,PASS


Precision QRIS exact          : 0.705882
Precision QRIS presentation   : 71%
Precision QRIS+EDC exact      : 0.833333
Precision QRIS+EDC presentation: 83%


## 6. Bangun portfolio non-BNI dari rekomendasi aktual

Pada tahap ini recommendation **tidak dibuat ulang secara manual**. Kita membaca hasil produksi Notebook 04, lalu hanya mengambil merchant dengan `is_bni_acquiring == "Tidak"`.

In [7]:
portfolio = product_rec.merge(
    mdr_rebuilt,
    on="merchant_id",
    how="left",
    validate="one_to_one",
)

target = portfolio.loc[portfolio["is_bni_acquiring"].eq("Tidak")].copy()

n_target = len(target)
n_qris = int(target["product_recommendation"].eq("QRIS").sum())
n_edc = int(target["product_recommendation"].eq("QRIS + EDC").sum())

portfolio_counts = pd.DataFrame({
    "Komponen": ["Merchant non-BNI", "Prediksi QRIS", "Prediksi QRIS + EDC"],
    "Nilai": [n_target, n_qris, n_edc],
})
display(portfolio_counts)

assert n_qris + n_edc == n_target, (
    "Ada merchant target tanpa salah satu dari dua recommendation class."
)

,Komponen,Nilai
0,Merchant non-BNI,246
1,Prediksi QRIS,98
2,Prediksi QRIS + EDC,148


## 7. Contribution per merchant dari variabel yang sudah direkonstruksi

Asumsi biaya produk pada economic layer:

- QRIS: **Rp50.000 / merchant / bulan**
- EDC incremental: **Rp200.000 / terminal / bulan**

Formula:

- **QRIS** → `monthly_net_qris_fbi - QRIS_COST`
- **QRIS + EDC** → `monthly_net_qris_fbi + monthly_net_edc_fbi - QRIS_COST - EDC_COST`

Cost adalah asumsi capstone; FBI berasal dari transaksi.

In [8]:
QRIS_COST = 50_000
EDC_COST = 200_000

is_edc_rec = target["product_recommendation"].eq("QRIS + EDC")

target["estimated_contribution"] = np.where(
    is_edc_rec,
    target["monthly_net_qris_fbi"]
    + target["monthly_net_edc_fbi"]
    - QRIS_COST
    - EDC_COST,
    target["monthly_net_qris_fbi"] - QRIS_COST,
)

edc_rec = target.loc[is_edc_rec].copy()
qris_rec = target.loc[target["product_recommendation"].eq("QRIS")].copy()

edc_group = {
    "n": len(edc_rec),
    "net_qris_fbi_total": edc_rec["monthly_net_qris_fbi"].sum(),
    "net_edc_fbi_total": edc_rec["monthly_net_edc_fbi"].sum(),
    "net_total_fbi": (
        edc_rec["monthly_net_qris_fbi"].sum()
        + edc_rec["monthly_net_edc_fbi"].sum()
    ),
    "cost_total": len(edc_rec) * (QRIS_COST + EDC_COST),
    "contribution_total": edc_rec["estimated_contribution"].sum(),
    "contribution_avg": edc_rec["estimated_contribution"].mean(),
}

qris_group = {
    "n": len(qris_rec),
    "net_qris_fbi_total": qris_rec["monthly_net_qris_fbi"].sum(),
    "cost_total": len(qris_rec) * QRIS_COST,
    "contribution_total": qris_rec["estimated_contribution"].sum(),
    "contribution_avg": qris_rec["estimated_contribution"].mean(),
}

edc_breakdown = pd.DataFrame([
    ["Net FBI QRIS", edc_group["net_qris_fbi_total"], edc_group["net_qris_fbi_total"] / edc_group["n"]],
    ["Net FBI EDC", edc_group["net_edc_fbi_total"], edc_group["net_edc_fbi_total"] / edc_group["n"]],
    ["Total Net FBI", edc_group["net_total_fbi"], edc_group["net_total_fbi"] / edc_group["n"]],
    ["Cost QRIS + EDC", edc_group["cost_total"], edc_group["cost_total"] / edc_group["n"]],
    ["Contribution", edc_group["contribution_total"], edc_group["contribution_avg"]],
], columns=["Langkah", f"Total {edc_group['n']} merchant", "Rata-rata / merchant"])

display(edc_breakdown)

qris_breakdown = pd.DataFrame([
    ["Net FBI QRIS", qris_group["net_qris_fbi_total"], qris_group["net_qris_fbi_total"] / qris_group["n"]],
    ["Cost QRIS", qris_group["cost_total"], qris_group["cost_total"] / qris_group["n"]],
    ["Contribution", qris_group["contribution_total"], qris_group["contribution_avg"]],
], columns=["Langkah", f"Total {qris_group['n']} merchant", "Rata-rata / merchant"])

display(qris_breakdown)

print(f"Avg contribution QRIS + EDC : Rp{edc_group['contribution_avg']:,.2f}/bulan")
print(f"Avg contribution QRIS       : Rp{qris_group['contribution_avg']:,.2f}/bulan")

,Langkah,Total 148 merchant,Rata-rata / merchant
0,Net FBI QRIS,"29,627,380.2529","200,185.0017"
1,Net FBI EDC,"64,602,341.2278","436,502.3056"
2,Total Net FBI,"94,229,721.4807","636,687.3073"
3,Cost QRIS + EDC,"37,000,000.0000","250,000.0000"
4,Contribution,"57,229,721.4807","386,687.3073"


,Langkah,Total 98 merchant,Rata-rata / merchant
0,Net FBI QRIS,"6,889,942.9741","70,305.5406"
1,Cost QRIS,"4,900,000.0000","50,000.0000"
2,Contribution,"1,989,942.9741","20,305.5406"


Avg contribution QRIS + EDC : Rp386,687.31/bulan
Avg contribution QRIS       : Rp20,305.54/bulan


## 8. Positive incremental EDC opportunity pada merchant predicted QRIS

Untuk merchant yang saat ini direkomendasikan QRIS, kita tetap dapat melihat **potensi economics EDC** dari observed payment mix.

Agar opportunity tidak dihitung ketika FBI EDC bahkan tidak menutup biaya terminal:

```text
positive_incremental_edc = max(monthly_net_edc_fbi - EDC_COST, 0)
```

Ini menjadi basis opportunity loss untuk approximation FN.

In [9]:
qris_rec["positive_incremental_edc"] = np.maximum(
    qris_rec["monthly_net_edc_fbi"] - EDC_COST,
    0.0,
)

positive_edc_pool = float(qris_rec["positive_incremental_edc"].sum())
positive_edc_n = int(qris_rec["positive_incremental_edc"].gt(0).sum())

positive_edc_table = pd.DataFrame({
    "Komponen": [
        "Predicted QRIS merchant",
        "Merchant dengan positive incremental EDC contribution",
        "Total positive incremental EDC contribution",
        f"Rata-rata dibagi seluruh {len(qris_rec)} predicted QRIS",
        f"Rata-rata hanya {positive_edc_n} merchant yang positif",
    ],
    "Nilai": [
        len(qris_rec),
        positive_edc_n,
        positive_edc_pool,
        positive_edc_pool / len(qris_rec) if len(qris_rec) else np.nan,
        positive_edc_pool / positive_edc_n if positive_edc_n else np.nan,
    ],
})
display(positive_edc_table)

print(f"Positive incremental EDC opportunity pool: Rp{positive_edc_pool:,.2f}/bulan")

,Komponen,Nilai
0,Predicted QRIS merchant,98.0000
1,Merchant dengan positive incremental EDC contr...,11.0000
2,Total positive incremental EDC contribution,"676,777.5004"
3,Rata-rata dibagi seluruh 98 predicted QRIS,"6,905.8929"
4,Rata-rata hanya 11 merchant yang positif,"61,525.2273"


Positive incremental EDC opportunity pool: Rp676,777.50/bulan


## 9. Estimasi TP / TN / FP / FN dari precision

Kita mulai dari **kelompok hasil prediksi**, sehingga class precision digunakan sebagai expected correct share pada masing-masing predicted group.

Dengan EDC sebagai positive class:

- Predicted `QRIS + EDC` yang benar → TP-like portfolio contribution.
- Predicted `QRIS + EDC` yang salah → FP-like incremental EDC cost.
- Predicted `QRIS` yang benar → TN-like portfolio contribution.
- Predicted `QRIS` yang salah → FN-like missed EDC opportunity.

Expected merchant boleh desimal karena ini **nilai harapan portfolio**, bukan label aktual individual merchant non-BNI.

In [10]:
expected_tp = n_edc * PRECISION_EDC_PRESENTATION
expected_fp = n_edc * (1 - PRECISION_EDC_PRESENTATION)
expected_tn = n_qris * PRECISION_QRIS_PRESENTATION
expected_fn = n_qris * (1 - PRECISION_QRIS_PRESENTATION)

expected_confusion = pd.DataFrame({
    "Kondisi": ["TP", "FP", "TN", "FN"],
    "Formula": [
        f"{n_edc} × {PRECISION_EDC_PRESENTATION:.0%}",
        f"{n_edc} × (1 - {PRECISION_EDC_PRESENTATION:.0%})",
        f"{n_qris} × {PRECISION_QRIS_PRESENTATION:.0%}",
        f"{n_qris} × (1 - {PRECISION_QRIS_PRESENTATION:.0%})",
    ],
    "Expected merchant": [expected_tp, expected_fp, expected_tn, expected_fn],
})
display(expected_confusion)

,Kondisi,Formula,Expected merchant
0,TP,148 × 83%,122.8400
1,FP,148 × (1 - 83%),25.1600
2,TN,98 × 71%,69.5800
3,FN,98 × (1 - 71%),28.4200


## 10. Final TP + TN vs FP + FN

### Business interpretation

- **TP contribution** = expected correct QRIS+EDC × rata-rata contribution kelompok predicted QRIS+EDC.
- **TN contribution** = expected correct QRIS × rata-rata contribution kelompok predicted QRIS.
- **FP cost** = expected wrong QRIS+EDC × **incremental EDC cost Rp200.000**. QRIS cost tidak dihitung lagi sebagai FP loss karena QRIS adalah base product di kedua kelas.
- **FN opportunity loss** = error share predicted QRIS × **positive incremental EDC opportunity pool**.
- **Expected Decision Value** = TP contribution + TN contribution − FP cost − FN opportunity loss.

Ini adalah **decision-support proxy**, bukan angka accounting resmi.

In [11]:
tp_contribution = expected_tp * edc_group["contribution_avg"]
tn_contribution = expected_tn * qris_group["contribution_avg"]
correct_contribution = tp_contribution + tn_contribution
fp_cost = expected_fp * EDC_COST
fn_opportunity_loss = (1 - PRECISION_QRIS_PRESENTATION) * positive_edc_pool
expected_decision_value = correct_contribution - fp_cost - fn_opportunity_loss

final_table = pd.DataFrame({
    "Komponen": [
        "TP contribution",
        "TN contribution",
        "Correct recommendation contribution",
        "FP cost",
        "FN opportunity loss",
        "Estimasi Nilai Keputusan Model",
    ],
    "Estimasi / bulan": [
        tp_contribution,
        tn_contribution,
        correct_contribution,
        fp_cost,
        fn_opportunity_loss,
        expected_decision_value,
    ],
})
display(final_table)

for _, row in final_table.iterrows():
    print(f"{row['Komponen']:<38} Rp{row['Estimasi / bulan']:,.0f}")

,Komponen,Estimasi / bulan
0,TP contribution,"47,500,668.8290"
1,TN contribution,"1,412,859.5116"
2,Correct recommendation contribution,"48,913,528.3406"
3,FP cost,"5,032,000.0000"
4,FN opportunity loss,"196,265.4751"
5,Estimasi Nilai Keputusan Model,"43,685,262.8654"


TP contribution                        Rp47,500,669
TN contribution                        Rp1,412,860
Correct recommendation contribution    Rp48,913,528
FP cost                                Rp5,032,000
FN opportunity loss                    Rp196,265
Estimasi Nilai Keputusan Model         Rp43,685,263


## 11. Rekonsiliasi dengan angka pada PDF

Angka di bawah **hanya menjadi reference target** untuk memeriksa apakah notebook sedang dijalankan pada versi artefak yang sama dengan PDF. Mereka **tidak digunakan sebagai input formula**.

Jika seluruh status `PASS`, berarti angka PDF berhasil direproduksi dari data + artefak model saat ini.

In [12]:
PDF_REFERENCE = {
    "Merchant non-BNI": 246,
    "Prediksi QRIS": 105,
    "Prediksi QRIS + EDC": 141,
    "Precision QRIS (presentation)": 0.72,
    "Precision QRIS + EDC (presentation)": 0.84,
    "Avg contribution QRIS + EDC": 397_819.40,
    "Avg contribution QRIS": 27_937.44,
    "Positive incremental EDC opportunity": 875_323.92,
    "TP contribution": 47_117_730,
    "TN contribution": 2_112_071,
    "Correct recommendation contribution": 49_229_800,
    "FP cost": 4_512_000,
    "FN opportunity loss": 245_091,
    "Expected Decision Value": 44_472_710,
}

CALCULATED = {
    "Merchant non-BNI": n_target,
    "Prediksi QRIS": n_qris,
    "Prediksi QRIS + EDC": n_edc,
    "Precision QRIS (presentation)": PRECISION_QRIS_PRESENTATION,
    "Precision QRIS + EDC (presentation)": PRECISION_EDC_PRESENTATION,
    "Avg contribution QRIS + EDC": edc_group["contribution_avg"],
    "Avg contribution QRIS": qris_group["contribution_avg"],
    "Positive incremental EDC opportunity": positive_edc_pool,
    "TP contribution": round(tp_contribution),
    "TN contribution": round(tn_contribution),
    "Correct recommendation contribution": round(correct_contribution),
    "FP cost": round(fp_cost),
    "FN opportunity loss": round(fn_opportunity_loss),
    "Expected Decision Value": round(expected_decision_value),
}

rows = []
for key, ref in PDF_REFERENCE.items():
    calc = CALCULATED[key]
    # Toleransi kecil untuk angka yang pada PDF dibulatkan.
    tol = 1.0 if abs(ref) >= 1 else 1e-12
    diff = float(calc) - float(ref)
    rows.append({
        "Metric": key,
        "Calculated from repo": calc,
        "PDF reference": ref,
        "Difference": diff,
        "Status": "PASS" if abs(diff) <= tol else "DIFFERENT VERSION / CHECK",
    })

pdf_reconciliation = pd.DataFrame(rows)
display(pdf_reconciliation)

n_pass = int(pdf_reconciliation["Status"].eq("PASS").sum())
print(f"PDF reconciliation: {n_pass}/{len(pdf_reconciliation)} metrics PASS")

if n_pass == len(pdf_reconciliation):
    print("PASS — Angka pada PDF berhasil direproduksi dari data dan artefak repository.")
else:
    print(
        "INFO — Ada angka yang berbeda dari PDF. Ini bisa valid bila Notebook 02/04 "
        "atau artefak rekomendasi sudah di-refit setelah PDF dibuat. Gunakan hasil calculated "
        "sebagai angka versi repo terbaru."
    )

,Metric,Calculated from repo,PDF reference,Difference,Status
0,Merchant non-BNI,246.0000,246.0000,0.0000,PASS
1,Prediksi QRIS,98.0000,105.0000,-7.0000,DIFFERENT VERSION / CHECK
2,Prediksi QRIS + EDC,148.0000,141.0000,7.0000,DIFFERENT VERSION / CHECK
3,Precision QRIS (presentation),0.7100,0.7200,-0.0100,DIFFERENT VERSION / CHECK
4,Precision QRIS + EDC (presentation),0.8300,0.8400,-0.0100,DIFFERENT VERSION / CHECK
5,Avg contribution QRIS + EDC,"386,687.3073","397,819.4000","-11,132.0927",DIFFERENT VERSION / CHECK
6,Avg contribution QRIS,"20,305.5406","27,937.4400","-7,631.8994",DIFFERENT VERSION / CHECK
7,Positive incremental EDC opportunity,"676,777.5004","875,323.9200","-198,546.4196",DIFFERENT VERSION / CHECK
8,TP contribution,"47,500,669.0000","47,117,730.0000","382,939.0000",DIFFERENT VERSION / CHECK
9,TN contribution,"1,412,860.0000","2,112,071.0000","-699,211.0000",DIFFERENT VERSION / CHECK


PDF reconciliation: 1/14 metrics PASS
INFO — Ada angka yang berbeda dari PDF. Ini bisa valid bila Notebook 02/04 atau artefak rekomendasi sudah di-refit setelah PDF dibuat. Gunakan hasil calculated sebagai angka versi repo terbaru.


## 12. Audit trail akhir

Jika notebook ini menghasilkan `PASS` pada rekonstruksi MDR/FBI dan reconciliation PDF, maka jalur pembuktiannya adalah:

```text
Observed C2M transactions
    ↓ amount × MDR rate
Gross MDR per transaction
    ↓ aggregate merchant / observation months
Monthly Gross MDR
    ↓ × 85%
Monthly Net FBI QRIS / EDC
    ↓ merge dengan product recommendation
Contribution per merchant
    ↓ class precision sebagai portfolio expected-correct share
Expected TP/TN contribution
    ↓ FP incremental EDC cost + FN missed EDC opportunity
Expected Decision Value
```

### Hal yang tidak boleh salah diinterpretasikan

- 246 merchant adalah **merchant non-BNI target**, bukan seluruh merchant graph.
- TP/TN/FP/FN untuk non-BNI adalah **portfolio approximation** karena actual product label target belum diketahui.
- Rp397.819 dan Rp27.937 adalah rata-rata contribution pada **kelompok predicted class**, bukan rata-rata actual TP/TN.
- FP cost hanya incremental EDC cost karena QRIS tetap base product di kedua kelas.
- FN loss adalah opportunity yang terlewat, bukan biaya EDC Rp200.000.
- Expected Decision Value adalah **decision-support proxy**, bukan accounting Net Fee-Based Income resmi.